# EODH Community Notebook Competition 2026

## Exploring a Sentinel-2 NDWI time series with the EO Data Hub

Discover Sentinel-2 Analysis Ready Data with pyeodh, sample a multiband COG and examine NDWI through time.</p>

* Author(s): EODH Team
* Organisation / affiliation: EODH
* Language: Python
*Date: 2026-10-02


## 1. Purpose

This notebook adapts the EO Data Hub training example [**Handling Timeseries Data**](https://docs.eodatahub.org.uk/training-materials/examples/timeseries/) into the Community Notebook Competition format. It demonstrates how to discover Sentinel-2 Analysis Ready Data (ARD), sample spectral bands directly from cloud-hosted imagery and create an NDWI time series.

**What the reader should learn:**
How to search the EODH Resource Catalogue with `pyeodh`, constrain results by location, date and cloud cover, inspect STAC assets and band metadata, sample a multiband COG and visualise a derived environmental time series.

### Objectives
1. Search the `sentinel2_ard` collection.
2. Find April–September 2024 scenes over the WWT London Wetland Centre with ≤30% cloud cover.
3. Identify B03 and B08 from the COG asset's `eo:bands` metadata.
4. Calculate NDWI at the sample point for each usable scene.
5. Plot and summarise the resulting time series.

### Expected outputs
A catalogue-search table, asset/band information, a table of NDWI observations, an interactive time-series plot and summary statistics.


## 2. Background and approach

The EODH `sentinel2_ard` collection contains Sentinel-2 imagery processed into surface-reflectance Analysis Ready Data. This example calculates the **Normalised Difference Water Index (NDWI)**:

NDWI = {B03-B08}/{B03+B08}

where B03 is green and B08 is near infrared. 

### Workflow overview
1. Connect to the EODH catalogue.
2. Retrieve the Sentinel-2 ARD collection.
3. Search by point, date and cloud cover.
4. Inspect a result and its STAC assets.
5. Transform the point from EPSG:4326 to EPSG:27700.
6. Read B03 and B08 from each remote COG.
7. Calculate and plot NDWI through time.

**Reproducibility:** The catalogue is live, so returned records may change. Scene-level cloud cover also does not guarantee that the sample point itself is cloud-free.


<h2 class="eodh-section">3. Data and other inputs</h2>

| Dataset / input | Provider | Access | Purpose |
|---|---|---|---|
| Sentinel-2 ARD (`sentinel2_ard`) | Defra / JNCC; distributed through CEDA/EODH | `public/catalogs/ceda-stac-catalogue` | Surface-reflectance time series |
| Sample point | WWT London Wetland Centre | Defined below | NDWI sampling location |

**Sample point (EPSG:4326):** −0.2346089199955088, 51.478446832015834  
**Period:** 1 April–30 September 2024  
**Maximum scene cloud cover:** 30%

The source example uses a point to demonstrate temporal sampling. A competition entry could instead use an AOI and calculate spatial statistics for each acquisition.


<h2 class="eodh-section">4. Environment and setup</h2>

This Python example is intended for the EODH Notebook Service. `pyeodh` provides catalogue access, `rasterio` reads the COGs, `pyproj` transforms coordinates, and `pandas`/`hvplot` organise and visualise the results.


In [ ]:
# Uncomment if required.
# %pip install --upgrade pyeodh
# %pip install hvplot shapely rasterio pyproj pandas
%pip install hvplot


In [4]:
import pyeodh
import shapely
import rasterio
import pandas as pd
import hvplot.pandas

from pyproj import Transformer
from IPython.display import display

BASE_URL = "https://eodatahub.org.uk"
CATALOG_PATH = "public/catalogs/ceda-stac-catalogue"
COLLECTION_ID = "sentinel2_ard"
START_DATE = "2024-04-01"
END_DATE = "2024-09-30"
MAX_CLOUD_COVER = 30

LON, LAT = (-0.2346089199955088, 51.478446832015834)
SAMPLE_POINT = shapely.Point(LON, LAT)


<h2 class="eodh-section">5. Discover Sentinel-2 ARD</h2>

### 5.1 Connect to the Resource Catalogue
`pyeodh.Client` is the entry point to EODH APIs. Here we retrieve the Sentinel-2 ARD collection from the public CEDA catalogue.


In [5]:
client = pyeodh.Client(base_url=BASE_URL).get_catalog_service()

s2ard = (
    client.get_catalog(CATALOG_PATH)
    .get_collection(COLLECTION_ID)
)

print("Collection ID:", s2ard.id)
print("Title:", s2ard.title)
print("Description:")
print(s2ard.description)


Collection ID: sentinel2_ard
Title: Sentinel 2 ARD
Description:
These data have been created by the Department for Environment, Food and Rural Affairs (Defra) and Joint Nature Conservation Committee (JNCC) in order to cost-effectively provide high quality, Analysis Ready Data (ARD) for a wide range of applications. The dataset contains modified Copernicus Sentinel-2 (Level 1C data processed into a surface reflectance product using ARCSI software (Level 2)).


### 5.2 Search by location, date and cloud cover

The search combines a spatial intersection with the WWT point, an April–September date window and a maximum reported scene cloud cover of 30%. The returned iterator is converted to a list so it can safely be counted and reused.


In [14]:
items = list(
    client.search(
        collections=[COLLECTION_ID],
        catalog_paths=[CATALOG_PATH],
        intersects=SAMPLE_POINT,
        query=[
            f"start_datetime>={START_DATE}",
            f"end_datetime<={END_DATE}",
            f"eo:cloud_cover<={MAX_CLOUD_COVER}",
        ],
    )
)

if not items:
    raise RuntimeError("No matching Sentinel-2 ARD scenes were found.")

print(f"Total matching items: {len(items)}")

search_summary = pd.DataFrame({
    "item_id": [item.id for item in items],
    "datetime": [item.datetime for item in items],
}).sort_values("datetime")

display(search_summary)


Total matching items: 13


,item_id,datetime
12,neodc.sentinel_ard.data.sentinel_2.2024.04.30....,2024-04-30 11:06:19+00:00
11,neodc.sentinel_ard.data.sentinel_2.2024.05.07....,2024-05-07 10:56:19+00:00
10,neodc.sentinel_ard.data.sentinel_2.2024.05.20....,2024-05-20 11:06:19+00:00
9,neodc.sentinel_ard.data.sentinel_2.2024.06.26....,2024-06-26 10:56:19+00:00
8,neodc.sentinel_ard.data.sentinel_2.2024.07.19....,2024-07-19 11:06:19+00:00
7,neodc.sentinel_ard.data.sentinel_2.2024.07.29....,2024-07-29 11:06:19+00:00
6,neodc.sentinel_ard.data.sentinel_2.2024.07.31....,2024-07-31 10:56:21+00:00
5,neodc.sentinel_ard.data.sentinel_2.2024.08.18....,2024-08-18 11:06:19+00:00
4,neodc.sentinel_ard.data.sentinel_2.2024.08.23....,2024-08-23 11:06:21+00:00
3,neodc.sentinel_ard.data.sentinel_2.2024.08.25....,2024-08-25 10:56:19+00:00


### 5.3 Inspect assets and spectral-band metadata

The ARD item contains several assets. The analysis uses the multiband `cog` asset. Its `eo:bands` metadata tells us which raster bands correspond to Sentinel-2 B03 and B08, avoiding hard-coded band positions.


In [15]:
example_item = items[0]

print("Example item:", example_item.id)
print(f"Cloud cover: {example_item.properties.get('eo:cloud_cover')}%")
print("\nAssets:")
for key, asset in example_item.assets.items():
    print(f"  {key:20s} {asset.href}")


Example item: neodc.sentinel_ard.data.sentinel_2.2024.09.19.S2A_20240919_latn518lonw0008_T30UXC_ORB094_20240919144023_utm30n_osgb
Cloud cover: None%

Assets:
  cloud                https://dap.ceda.ac.uk/neodc/sentinel_ard/data/sentinel_2/2024/09/19/S2A_20240919_latn518lonw0008_T30UXC_ORB094_20240919144023_utm30n_osgb_clouds.tif
  cloud_probability    https://dap.ceda.ac.uk/neodc/sentinel_ard/data/sentinel_2/2024/09/19/S2A_20240919_latn518lonw0008_T30UXC_ORB094_20240919144023_utm30n_osgb_clouds_prob.tif
  metadata             https://dap.ceda.ac.uk/neodc/sentinel_ard/data/sentinel_2/2024/09/19/S2A_20240919_latn518lonw0008_T30UXC_ORB094_20240919144023_utm30n_osgb_vmsk_sharp_rad_srefdem_stdsref_meta.xml
  thumbnail            https://dap.ceda.ac.uk/neodc/sentinel_ard/data/sentinel_2/2024/09/19/S2A_20240919_latn518lonw0008_T30UXC_ORB094_20240919144023_utm30n_osgb_vmsk_sharp_rad_srefdem_stdsref_thumbnail.jpg
  topographic_shadow   https://dap.ceda.ac.uk/neodc/sentinel_ard/data/sentinel_2/2

In [16]:
cog_asset = example_item.assets["cog"]
bands = cog_asset.extra_fields["eo:bands"]
display(pd.DataFrame(bands))

band_index = {band["name"]: i + 1 for i, band in enumerate(bands)}
print("B03 raster band:", band_index["B03"])
print("B08 raster band:", band_index["B08"])

,eo: full_width_half_max,name,eo:central_wavelength,description,eo:common_name
0,0.07,B02,496.6,Blue,blue
1,0.04,B03,560.0,Green,green
2,0.03,B04,664.5,Red,red
3,0.02,B05,703.9,Visible and Near Infrared,rededge
4,0.02,B06,740.2,Visible and Near Infrared,rededge
5,0.02,B07,782.5,Visible and Near Infrared,rededge
6,0.11,B08,835.1,Visible and Near Infrared,nir
7,0.02,B08a,864.8,Visible and Near Infrared,nir08
8,0.09,B11,1613.7,Short Wave Infrared,swir16
9,0.18,B12,2202.4,Short Wave Infrared,swir22


B03 raster band: 2
B08 raster band: 7


## 6. Build the NDWI time series

The catalogue point is in EPSG:4326, while the Sentinel-2 ARD raster is in British National Grid. The point is transformed to EPSG:27700 before sampling.

For every result, the code opens the remote COG, obtains the B03/B08 positions from `eo:bands`, samples both bands and calculates NDWI. 

In [18]:
transformer = Transformer.from_crs(
    "EPSG:4326", "EPSG:27700", always_xy=True
)
x, y = transformer.transform(LON, LAT)
print(f"EPSG:27700 sample point: x={x:.2f}, y={y:.2f}")

records = []

for item in items:
    if "cog" not in item.assets:
        continue

    asset = item.assets["cog"]
    bands = asset.extra_fields.get("eo:bands", [])
    band_index = {band["name"]: i + 1 for i, band in enumerate(bands)}

    if "B03" not in band_index or "B08" not in band_index:
        continue

    try:
        with rasterio.open(asset.href) as src:
            green = next(src.sample([(x, y)], indexes=band_index["B03"]))[0]
            nir = next(src.sample([(x, y)], indexes=band_index["B08"]))[0]
            nodata = src.nodata
    except (rasterio.errors.RasterioIOError, OSError, ValueError) as exc:
        print(f"Skipping {item.id}: {exc}")
        continue

    if nodata is not None and (green == nodata or nir == nodata):
        continue
    if green <= 0 or nir <= 0:
        continue

    denominator = float(green) + float(nir)
    if denominator == 0:
        continue

    records.append({
        "date": pd.to_datetime(item.datetime),
        "ndwi": (float(green) - float(nir)) / denominator,
        "item_id": item.id,
    })

df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)

if df.empty:
    raise RuntimeError("No valid NDWI observations were produced.")

display(df)


EPSG:27700 sample point: x=522692.71, y=176976.07


,date,ndwi,item_id
0,2024-04-30 11:06:19+00:00,-0.685345,neodc.sentinel_ard.data.sentinel_2.2024.04.30....
1,2024-05-07 10:56:19+00:00,-0.696356,neodc.sentinel_ard.data.sentinel_2.2024.05.07....
2,2024-05-20 11:06:19+00:00,-0.719921,neodc.sentinel_ard.data.sentinel_2.2024.05.20....
3,2024-06-26 10:56:19+00:00,-0.794872,neodc.sentinel_ard.data.sentinel_2.2024.06.26....
4,2024-07-19 11:06:19+00:00,-0.775401,neodc.sentinel_ard.data.sentinel_2.2024.07.19....
5,2024-07-29 11:06:19+00:00,-0.767956,neodc.sentinel_ard.data.sentinel_2.2024.07.29....
6,2024-07-31 10:56:21+00:00,-0.708333,neodc.sentinel_ard.data.sentinel_2.2024.07.31....
7,2024-08-18 11:06:19+00:00,-0.205229,neodc.sentinel_ard.data.sentinel_2.2024.08.18....
8,2024-08-23 11:06:21+00:00,-0.549020,neodc.sentinel_ard.data.sentinel_2.2024.08.23....
9,2024-08-25 10:56:19+00:00,-0.640832,neodc.sentinel_ard.data.sentinel_2.2024.08.25....


## 7. Results and visualisation

The interactive plot shows NDWI at the sample point for each usable Sentinel-2 observation. Individual values may still be affected by local cloud, shadow or mixed pixels even after filtering by scene-level cloud cover.


In [19]:
df.hvplot.line(
    x="date", y="ndwi",
    title="NDWI time series at WWT London Wetland Centre",
    ylabel="NDWI", xlabel="Date",
    grid=True, width=750, height=380,
) * df.hvplot.scatter(
    x="date", y="ndwi",
    size=70,
    hover_cols=["cloud_cover", "item_id"],
)


:Overlay
   .Curve.I   :Curve   [date]   (ndwi)
   .Scatter.I :Scatter   [date]   (ndwi,item_id)

### Summary statistics

These statistics describe the valid temporal observations at the sample point.


In [10]:
print("NDWI TIME-SERIES SUMMARY")
print("=" * 40)
print(f"Valid observations : {len(df)}")
print(f"First observation  : {df['date'].min().date()}")
print(f"Last observation   : {df['date'].max().date()}")
print(f"Minimum NDWI       : {df['ndwi'].min():.4f}")
print(f"Maximum NDWI       : {df['ndwi'].max():.4f}")
print(f"Mean NDWI          : {df['ndwi'].mean():.4f}")
print(f"Median NDWI        : {df['ndwi'].median():.4f}")
print(f"Std. deviation     : {df['ndwi'].std():.4f}")


NDWI TIME-SERIES SUMMARY
Valid observations : 13
First observation  : 2024-04-30
Last observation   : 2024-09-19
Minimum NDWI       : -0.7949
Maximum NDWI       : -0.1275
Mean NDWI          : -0.6200
Median NDWI        : -0.6964
Std. deviation     : 0.2115


## 8. EO Data Hub capability demonstrated

This notebook demonstrates:

- **Resource discovery with `pyeodh`** using collection, spatial, temporal and metadata constraints.
- **STAC metadata** to discover assets and spectral-band positions.
- **Cloud-hosted data access** by sampling remote COGs rather than downloading every complete scene.
- **Analysis Ready Data** for repeated surface-reflectance observations.
- **Interactive analysis** using pandas and hvPlot.

**Why the Hub is useful for this workflow:** catalogue discovery, access to large EO holdings and analysis happen in one environment, while the user only reads the small amount of source imagery needed for the analysis.


## 9. Conclusions and next steps

This example turns a catalogue search into a simple environmental time series without first downloading a stack of complete Sentinel-2 scenes.

### Limitations
The cloud threshold applies to the complete scene rather than the sample point. The example does not explicitly apply the ARD cloud, valid-pixel or topographic-shadow masks, and a single point can be sensitive to mixed pixels.

### Possible extensions
Use an AOI instead of one point, apply the available quality masks, calculate a spatial median for each date, compare multiple sites, extend the time period, or scale the workflow using EODH compute services.


## References and acknowledgements

### References
- UK EO Data Hub documentation, [**Handling Timeseries Data**](https://docs.eodatahub.org.uk/training-materials/examples/timeseries/) — technical source example adapted here.
- UK EO Data Hub documentation, [**pyeodh**](https://docs.eodatahub.org.uk/Developers/tools/pyeodh/).

### Acknowledgements
The Sentinel-2 ARD collection was created by Defra and JNCC from Copernicus Sentinel-2 data and is made accessible through CEDA/EODH. 


## Copyright and licensing

By submitting this notebook, the author(s) confirm that they have the right to publish the submitted material and are content for the methods and intellectual property disclosed in the submission to be publicly accessible under the licences stated below.

### Code

Unless a code cell or third-party component is explicitly identified otherwise, **original code written for this notebook is licensed under the MIT License**.

**Copyright:** © 2026 EODH


### Notebook text, original figures and other original non-code content

Unless otherwise stated, original non-code content created by the author(s) for this notebook is licensed under the **Creative Commons Attribution 4.0 International (CC BY 4.0)** licence.

### Author declaration

By submitting this notebook, we confirm that, to the best of our knowledge:

- we have permission to publish all material included in the submission;
- the licensing and attribution information supplied is accurate;
- no confidential information, credentials or material that cannot lawfully be made public has been included;
- third-party and commercial/proprietary licence conditions have been followed, including conditions applying to derived works; and
- we understand that the submitted notebook, its methods and the intellectual property intentionally disclosed within it will be publicly accessible.


<div style="margin-top:38px;padding:26px 20px 12px 20px;border-top:2px solid #dbe2ee;text-align:center;font-family:Arial,Helvetica,sans-serif;color:#263d73;">
<strong>EARTH OBSERVATION DATA HUB</strong>
<p style="margin-top:18px;font-size:15px;font-weight:600;text-align:center">Notebook created as part of the EODH Community Notebook Competition 2026.</p>
</div>
